In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/doc-forge-task-3-bank-loan-default-prediction/sample_submission.csv
/kaggle/input/competitions/doc-forge-task-3-bank-loan-default-prediction/train.csv
/kaggle/input/competitions/doc-forge-task-3-bank-loan-default-prediction/test.csv


In [2]:
import math
import numpy as np
import pandas as pd

In [3]:
## IMPORTING TEST FILE

In [4]:
import os
os.listdir('/kaggle/input')
os.listdir('/kaggle/input/competitions/doc-forge-task-3-bank-loan-default-prediction')

['sample_submission.csv', 'train.csv', 'test.csv']

# LOADING THE DATASET 
like always we are first using df.head() to print the first 5 rows of the table

Attributes present in the table are:
1. ID
2. Age
3. Annual_Income
4. Credit_Score
5. Debt_to_Income
6. Employment_Years
7. Previous_Defaults
8. Education_Level
9. is_default

2 Important Observation(Wrt to our ML model):
* REMOVING ID - Its useless for prediction
* TARGET - is_default (This is the target (label) that the network tries to predict)
  

In [5]:
train = pd.read_csv('/kaggle/input/competitions/doc-forge-task-3-bank-loan-default-prediction/train.csv')
train.head()

,ID,Age,Annual_Income,Credit_Score,Loan_Amount,Debt_to_Income,Employment_Years,Previous_Defaults,Education_Level,is_default
0,16868,31,74956.384288,696,30521.270737,0.249670,8,0,0,0
1,16107,58,53338.688256,496,27132.886897,0.493917,27,0,3,1
2,14367,46,15000.000000,783,4301.476830,0.241996,34,0,3,0
3,23009,35,57895.729791,738,39988.650926,0.574912,40,0,0,0
4,14539,56,71309.519094,785,16439.459217,0.658744,13,0,1,0


In [6]:
test = pd.read_csv('/kaggle/input/competitions/doc-forge-task-3-bank-loan-default-prediction/test.csv')

Basic exploratory analysis is performed to examine the shape, data types, and statistical properties of the dataset. This provides insights into feature distributions and potential preprocessing requirements.

In [7]:
train.shape

(10500, 10)

In [8]:
train.info

<bound method DataFrame.info of           ID  Age  Annual_Income  Credit_Score   Loan_Amount  Debt_to_Income  \
0      16868   31   74956.384288           696  30521.270737        0.249670   
1      16107   58   53338.688256           496  27132.886897        0.493917   
2      14367   46   15000.000000           783   4301.476830        0.241996   
3      23009   35   57895.729791           738  39988.650926        0.574912   
4      14539   56   71309.519094           785  16439.459217        0.658744   
...      ...  ...            ...           ...           ...             ...   
10495  23363   34   30369.726999           777  39749.285181        0.564227   
10496  17627   25   62464.380305           725   4436.013872        0.290957   
10497  22351   43   74663.361831           553  11876.722383        0.635125   
10498  23448   60   69000.449571           435  26021.292905        0.594122   
10499  15717   69   65943.187063           583   2000.000000        0.548145   

       

There is no missing data ie all columns have 10,500 values

# 📈STATISTICAL READINGS

In [9]:
train.describe()

,ID,Age,Annual_Income,Credit_Score,Loan_Amount,Debt_to_Income,Employment_Years,Previous_Defaults,Education_Level,is_default
count,10500.000000,10500.000000,10500.000000,10500.000000,10500.000000,10500.000000,10500.000000,10500.000000,10500.000000,10500.000000
mean,17536.792286,44.885619,55186.314067,573.953619,20180.831651,0.375389,21.976762,0.239429,1.506381,0.543048
std,4342.775915,14.072139,19711.734556,157.281457,9640.570922,0.188678,13.041109,0.526476,1.115083,0.498167
min,10001.000000,21.000000,15000.000000,300.000000,2000.000000,0.050114,0.000000,0.000000,0.000000,0.000000
25%,13763.750000,33.000000,41234.511889,438.000000,13273.325583,0.213157,11.000000,0.000000,1.000000,0.000000
50%,17537.000000,45.000000,55118.557922,573.000000,20010.995828,0.371561,22.000000,0.000000,1.000000,1.000000
75%,21312.250000,57.000000,68713.771766,707.000000,26721.898390,0.542108,33.000000,0.000000,3.000000,1.000000
max,25000.000000,69.000000,137957.901007,849.000000,55743.008612,0.699930,44.000000,2.000000,3.000000,1.000000


# CHECKING MISSING VALUES

ID is removed as it does not contribute to prediction

In [10]:
train.isnull().sum()

ID                   0
Age                  0
Annual_Income        0
Credit_Score         0
Loan_Amount          0
Debt_to_Income       0
Employment_Years     0
Previous_Defaults    0
Education_Level      0
is_default           0
dtype: int64

# UNIQUE VALUES
How many unique values per column


In [11]:
train.nunique

<bound method DataFrame.nunique of           ID  Age  Annual_Income  Credit_Score   Loan_Amount  Debt_to_Income  \
0      16868   31   74956.384288           696  30521.270737        0.249670   
1      16107   58   53338.688256           496  27132.886897        0.493917   
2      14367   46   15000.000000           783   4301.476830        0.241996   
3      23009   35   57895.729791           738  39988.650926        0.574912   
4      14539   56   71309.519094           785  16439.459217        0.658744   
...      ...  ...            ...           ...           ...             ...   
10495  23363   34   30369.726999           777  39749.285181        0.564227   
10496  17627   25   62464.380305           725   4436.013872        0.290957   
10497  22351   43   74663.361831           553  11876.722383        0.635125   
10498  23448   60   69000.449571           435  26021.292905        0.594122   
10499  15717   69   65943.187063           583   2000.000000        0.548145   

    

MISSING VALUE CHECK
We are using train.isnull().sum() to do this.

OBSERVATIONS:
- No missing values are present across the dataset.
- So there is no imputation required.
- The 'ID' column is removed as it does not contribute to prediction.

In [12]:
#No ID can be seen here
train.columns

Index(['ID', 'Age', 'Annual_Income', 'Credit_Score', 'Loan_Amount',
       'Debt_to_Income', 'Employment_Years', 'Previous_Defaults',
       'Education_Level', 'is_default'],
      dtype='object')

# Splitting Features and Target

The dataset is split into:
* Features (X)
* Target variable (y)
* Target : is_default
  

In [13]:
X = train.drop('is_default', axis=1)
y = train['is_default']

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
# Save feature columns BEFORE scaling (IMPORTANT)
feature_cols = train.drop(columns=['is_default']).columns

# Remove ID if exists
test = test.drop(columns=['ID'], errors='ignore')

# Fill missing values
test = test.fillna(test.mean())

# Align test to training features
test = test.reindex(columns=feature_cols, fill_value=0)

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
test_scaled = scaler.transform(test)

# BUILDING THE NEURAL NETWORK

Importing Deep Learning Libraries:
1) KERAS -Keras is a high-level API in TensorFlow that simplifies building and training neural network models.
2) Tensor Flow - TensorFlow is an open-source machine learning framework used to build and train neural networks efficiently.
3) Layers - These are the building blocks of a neural network that process input data and learn patterns through weights and activations.

In [17]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2026-04-10 21:00:23.604289: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775854823.868725      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775854823.945309      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775854824.550283      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775854824.550345      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775854824.550348      16 computation_placer.cc:177] computation placer alr

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

### Architecture:
- **Input Layer → 32 neurons (ReLU)**
- **Hidden Layer → 16 neurons (ReLU)**
- **Output Layer → 1 neuron (Sigmoid)**

Explanation - Basically Neural networks can be looked in the form of thousands of neurons (functions) connected with each other in a complex manner. We have considered 3 columns of such neurons wherein the input layer as 32 neurons, the hidden layer having 16 and the final sigmoid layer having just 1 ie is_default.

* ReLU activation - Rectified Linear Unit. ReLU allows positive values to pass through unchanged while setting all negative values to zero.

* Sigmoid Activation - a mathematical function that maps any real-valued input to a probability value between 0 and 1, creating an S-shaped curve. This is done in logistical regression using function y = 1/ 1 + e^x



In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential()

model.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))

model.add(Dense(16, activation='relu'))

model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-04-10 21:00:52.133880: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [20]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

Here we are configuring the learning process of the network using model.complile

* ADAM - Works as optimizer
* binary_crossentropy - Works as Loss Function
* accuracy - Works as Metrics

In [21]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data = (X_val, y_val)
)

Epoch 1/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7582 - loss: 0.5149 - val_accuracy: 0.9286 - val_loss: 0.1863
Epoch 2/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9274 - loss: 0.1737 - val_accuracy: 0.9305 - val_loss: 0.1649
Epoch 3/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9308 - loss: 0.1576 - val_accuracy: 0.9300 - val_loss: 0.1627
Epoch 4/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9313 - loss: 0.1541 - val_accuracy: 0.9329 - val_loss: 0.1629
Epoch 5/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9339 - loss: 0.1513 - val_accuracy: 0.9348 - val_loss: 0.1627
Epoch 6/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9361 - loss: 0.1407 - val_accuracy: 0.9357 - val_loss: 0.1633
Epoch 7/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9290 - loss: 0.1529 - val_accuracy: 0.9357 - val_loss: 0.1617
Epoch 8/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9379 - loss: 0.1377 - val_accuracy: 0.

The model shows consistent training and validation accuracy (approx 93%), indicating good generalization with minimal overfitting

In [22]:
model.evaluate(X_val, y_val)

66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9309 - loss: 0.1591


[0.16651038825511932, 0.9338095188140869]

# MAKING PREDICTIONS

In [23]:
#For validation data to check performance
y_pred_prob = model.predict(X_val)
y_pred = (y_pred_prob > 0.5).astype(int)

66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [24]:
#For final test predictions
preds = model.predict(test_scaled)
preds = (preds > 0.5).astype(int)

141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [25]:
test = pd.read_csv('/kaggle/input/competitions/doc-forge-task-3-bank-loan-default-prediction/test.csv')
test.head()

,ID,Age,Annual_Income,Credit_Score,Loan_Amount,Debt_to_Income,Employment_Years,Previous_Defaults,Education_Level
0,20261,68,91942.859904,579,19126.798380,0.387477,24,0,2
1,10770,47,63318.085405,429,35128.350201,0.071548,18,1,3
2,15050,52,43707.073009,522,18282.065259,0.230868,25,0,2
3,11773,44,47491.775237,832,36663.287658,0.293063,0,1,0
4,18588,51,26282.421850,673,29029.653536,0.426611,17,0,0


In [26]:
print(X.shape)
print(test.shape)

(10500, 9)
(4500, 9)


# ADDING F1 Score for Evaluation Metric

In [27]:
from sklearn.metrics import f1_score
y_pred = model.predict(X_val)
y_pred = (y_pred > 0.5).astype(int)
f1 = f1_score(y_val, y_pred)
print("F1 Score:", f1)

66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
F1 Score: 0.9394862864606008


In [28]:
# Load sample submission
sample_submission = pd.read_csv('/kaggle/input/competitions/doc-forge-task-3-bank-loan-default-prediction/sample_submission.csv')

# Get IDs
test_ids = sample_submission['ID']

# Create submission
submission = pd.DataFrame({
    'ID': test_ids,
    'is_default': preds.flatten()
})

# Save file
submission.to_csv('submission.csv', index=False)

In [29]:
print("Saving submission file...")
submission.to_csv('submission.csv', index=False)

import os
print("Files in directory:", os.listdir())

Saving submission file...
Files in directory: ['submission.csv', '__notebook__.ipynb']
